In [ ]:
# Install Tiny-SIMA and its dependencies into the active Jupyter kernel.
import os
import shutil
import subprocess
import sys
from pathlib import Path

jax_memory_fraction = os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.60"),
)

if not ((3, 10) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(
        "Tiny-SIMA currently requires a Python 3.10, 3.11, or 3.12 kernel."
    )

search_roots = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in search_roots if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Could not find pyproject.toml. Open this notebook from the cloned "
        "Tiny-SIMA repository."
    )

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode
        == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

install_target = f"{repo_root}[cuda12]" if has_cuda else str(repo_root)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "-e",
        install_target,
    ]
)

# Editable installs use a .pth file that is normally loaded at interpreter
# startup. Add src/ explicitly so this already-running kernel can import it.
source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print(f"Installed Tiny-SIMA from {repo_root}")
print(f"Added {source_root} to this kernel's import path")
print(f"CUDA dependencies requested: {has_cuda}")
print(f"JAX memory fraction: {jax_memory_fraction}")

# Train the Phase 1 baseline

By default this notebook runs the one-million-step PPO baseline. Set `RUN_1M = False` below to return to the short smoke configuration.

Before a long GPU run, shut down other notebooks using JAX and restart this kernel.

In [ ]:
from pathlib import Path

import jax

from tiny_sima.config import load_config
from tiny_sima.executor.ppo import train

print("JAX devices:", jax.devices())

In [ ]:
repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)

RUN_1M = True
config_name = "ppo_1m.yaml" if RUN_1M else "ppo_debug.yaml"
run_name = "ppo-1m" if RUN_1M else "notebook-baseline"
config = load_config(repo_root / "configs" / config_name).with_overrides(
    output_dir=str(repo_root / "runs" / run_name),
)
config.to_dict()

In [ ]:
summary = train(config)
summary